# Chamfer count sensitivity to λ

For each method and sequence, the lower score is better.

$$S(\lambda)=G+\lambda B,\qquad B=\frac{\sum_{i,p}|n^{pred}_{i,p}-n^{ref}_{i,p}|}{N_{pred}+N_{ref}}.$$

$G$ is full bidirectional nearest-neighbor Chamfer distance in $(x,y,vt)$, averaged equally over shared polarities. The benchmark uses $v=1$ pixel/ms. $B$ compares ON and OFF event counts separately in each RGB frame interval $i$, normalized by the combined event count. At $\lambda=0$ you see geometry alone. A steeper line means a larger count penalty, and a crossing means two methods exchange ranks.

Use one **chamfer_count** results CSV. The plots reuse its saved components and require no GPU.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

CSV_PATH = None  # Or set an explicit path to a chamfer_count results.csv
FULL_CSV_PATH = None  # Optional separate chamfer_full results.csv from the same run settings
LAMBDAS = [0, 0.5, 1, 2, 5]
SAMPLE = None  # First available sequence, or a name such as "hand_1"

In [ ]:
if CSV_PATH is None:
    root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "benchmarks" / "run_metric.py").is_file()), Path.cwd())
    candidates = sorted(root.glob("reports/hqevfi/chamfer_count/test-full-*/results.csv"))
    if len(candidates) != 1:
        raise ValueError("Set CSV_PATH to one chamfer_count CSV. Candidates: " + str(candidates))
    path = candidates[0]
else:
    path = Path(CSV_PATH).expanduser()

weights = np.unique(np.asarray(LAMBDAS, dtype=float))
if not weights.size or not np.isfinite(weights).all() or (weights < 0).any():
    raise ValueError("LAMBDAS must contain finite, nonnegative numbers.")
raw = pd.read_csv(path)
components = ["chamfer_geometry", "count_penalty"]
valid = raw.loc[raw["status"].eq("ok")].dropna(subset=components).copy()
if valid.empty or valid.duplicated(["sequence", "method"]).any():
    raise ValueError("Expected one valid row per sequence and method.")
scores = valid.set_index(["sequence", "method"])
scores = pd.DataFrame({w: scores.chamfer_geometry + w * scores.count_penalty for w in weights})
scores.columns.name = "λ"
sequences = sorted(valid.sequence.unique())
print(f"{path}\n{len(valid)} valid rows of {len(raw)}. Available sequences:")
print(", ".join(sequences))

In [ ]:
# One selection controls all plots and tables. Means use a fixed set of sequences.
methods = sorted(valid.method.unique())
checks = {m: widgets.Checkbox(value=True, description=m, indent=False,
                              layout=widgets.Layout(width="205px")) for m in methods}
display(widgets.Label("Show methods"), widgets.Box(list(checks.values()),
        layout=widgets.Layout(display="flex", flex_flow="row wrap")))

coverage = valid.groupby("sequence").method.nunique()
shared = coverage.index[coverage.eq(len(methods))]
mean_rows = scores.index.get_level_values("sequence").isin(shared)
mean_scores = scores.loc[mean_rows].groupby("method").mean()
sequence = sequences[0] if SAMPLE is None else SAMPLE
if sequence not in sequences:
    raise ValueError(f"Choose SAMPLE from {sequences}")
sample_scores = scores.xs(sequence, level="sequence").sort_index()
colors = dict(zip(methods, plt.get_cmap("tab20")(np.linspace(0, 1, len(methods)))))

def show_scores(table, title, selected):
    table = table.loc[table.index.isin(selected)]
    if table.empty:
        print(f"{title} — no selected methods with data.")
        return
    display(table.round(4))
    fig, ax = plt.subplots(figsize=(9, 4))
    for method, row in table.iterrows():
        ax.plot(weights, row, marker="o", color=colors[method], label=method)
    ax.set(xlabel="Count weight λ", ylabel="Score (pixel-equivalent)", title=title)
    ax.grid(alpha=0.25)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
    fig.tight_layout()
    plt.show()
    plt.close(fig)

In [ ]:
def draw_sweeps(**enabled):
    selected = [m for m in methods if enabled[m]]
    show_scores(mean_scores, f"Mean across {len(shared)} shared sequences", selected)
    show_scores(sample_scores, f"Sequence: {sequence}", selected)

display(widgets.interactive_output(draw_sweeps, checks))

In [ ]:
# Compare Full and Count at one λ. Delta is Count minus Full.
full = valid.set_index(["sequence", "method"])["chamfer_geometry"]
if FULL_CSV_PATH is not None:
    raw_full = pd.read_csv(Path(FULL_CSV_PATH).expanduser())
    actual = raw_full.loc[raw_full.status.eq("ok")].set_index(["sequence", "method"])["value"]
    if not actual.index.is_unique:
        raise ValueError("Duplicate sequence/method rows in Full CSV.")
    actual = actual.reindex(full.index)
    if not np.allclose(actual, full, rtol=1e-6, atol=1e-8):
        raise ValueError("Full CSV is missing matched rows or differs from the saved geometry. Check run settings.")
    full = actual
    print("Full source: separate CSV, matched by sequence and method.")
else:
    print("Full source: chamfer_geometry saved in the Count CSV.")

weight_picker = widgets.Dropdown(options=weights.tolist(), value=1.0 if 1 in weights else weights[0],
                                 description="Count λ")
scope_picker = widgets.ToggleButtons(options=["Mean", "Sample"], description="Compare")
display(widgets.HBox([scope_picker, weight_picker]))

def compare_full(scope, weight, **enabled):
    if scope == "Mean":
        baseline = full.loc[mean_rows].groupby("method").mean()
        count = mean_scores[weight]
        title = f"Mean across {len(shared)} shared sequences"
    else:
        baseline = full.xs(sequence, level="sequence")
        count = sample_scores[weight]
        title = sequence
    table = pd.DataFrame({"Chamfer Full": baseline, "Chamfer Count": count})
    table = table.loc[table.index.isin([m for m in methods if enabled[m]])]
    if table.empty:
        print("No selected methods with data.")
        return
    table["Delta"] = table["Chamfer Count"] - table["Chamfer Full"]
    display(table.round(4))
    ax = table[["Chamfer Full", "Chamfer Count"]].plot.barh(figsize=(8, max(2, len(table) * 0.55)))
    ax.set(xlabel="Score (pixel-equivalent)", ylabel="", title=f"{title} | λ = {weight:g}")
    ax.figure.tight_layout()
    plt.show()
    plt.close(ax.figure)

display(widgets.interactive_output(compare_full, dict(scope=scope_picker, weight=weight_picker, **checks)))